# Dense scientific ToT retrieval

Sentence Transformers supplies pretrained encoding; FAISS supplies indexing and search. The canonical adapter and the Phase 3 evaluator preserve corpus IDs and qrels. No model is fine-tuned.

From the repository root:
```sh
uv sync --locked
uv run python -m ipykernel install --sys-prefix --name scientific-tot --display-name "Scientific ToT"
uv run jupyter lab
```
Select **Scientific ToT** for both notebooks. The default input is the restored data at `../multimodal-tip-of-tongue-for-scientific-papers/data`. Set `TOT_DATA_ROOT` if your data lives elsewhere. By default, all available documents in the selected domain and all test queries are used with fulltext. For the two-document smoke test, set `TOT_DATA_ROOT=tests/fixtures/adapter` when launching from the repository root. The default output is a fresh directory per configuration-cell execution; an explicit output environment variable must name a fresh directory. The first BM25 run downloads Terrier's JAR; the first dense run downloads model weights. Java is supplied by the locked environment automatically.


In [ ]:
import json
import os
import sys
from pathlib import Path
from uuid import uuid4

from scientific_tot import Dataset, DatasetAdapter
from scientific_tot.integrations.dense import EncodingConfig, dense_experiment
from scientific_tot.integrations.selection import load_experiment_data

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
if Path(sys.prefix).resolve() != (ROOT / ".venv").resolve():
    raise RuntimeError("Select the Scientific ToT kernel registered from this project's .venv")
DATA = Path(
    os.environ.get(
        "TOT_DATA_ROOT", str(ROOT.parent / "multimodal-tip-of-tongue-for-scientific-papers/data")
    )
)

DATA = DATA.expanduser()
if not DATA.is_absolute():
    DATA = ROOT / DATA

OUTPUT = Path(
    os.environ.get("TOT_DENSE_OUTPUT", str(ROOT / "results" / f"dense-notebook-{uuid4().hex[:8]}"))
)
CACHE = Path(os.environ.get("TOT_DENSE_CACHE", str(ROOT / "results/dense-cache")))
BASELINE = os.environ.get("TOT_BM25_BASELINE")
DOMAIN = "ACL"
MISSING = "error"  # "skip" explicitly excludes incomplete records.
DOCUMENTS, QUERIES = None, None  # All documents/test queries; use 250, 10 for a demo.
CORPUS_SPLIT = "all"  # Include every available extraction, including distractors.
config = EncodingConfig(device="cpu", batch_size=16)
print("Python:", sys.executable, "Data:", DATA, "Output:", OUTPUT)
config

In [ ]:
adapter = DatasetAdapter(Dataset(local_dir=DATA))
corpora, queries = load_experiment_data(
    adapter,
    domain=DOMAIN,
    corpus_split=CORPUS_SPLIT,
    missing=MISSING,
    documents=DOCUMENTS,
    query_limit=QUERIES,
)
corpus = corpora["fulltext"]
print(len(corpus), "documents", len(queries), "queries")

## Substitute a pretrained model

Replace `config` before running below. For example:
```python
config = EncodingConfig(
    model_id="sentence-transformers/all-MiniLM-L6-v2",
    model_revision="main",  # Resolved once to a commit SHA and recorded.
    profile="model", device="cpu", batch_size=16,
)
```
The default E5 model automatically receives `query: ` and `passage: ` through the library prompt API. Other models use `encode_query`/`encode_document` with saved prompts and task routing. When a model card requires instructions absent from its configuration, supply `query_prompt`, `document_prompt`, or their named prompt settings explicitly. E5 instruct variants require a task-specific query prompt. Match the model's cosine/dot setting. Long text is truncated by the model; no chunking is performed.

In [ ]:
metrics = dense_experiment(
    corpus, queries, OUTPUT / "fulltext", cache=CACHE, config=config, baseline=BASELINE
)
metrics

In [ ]:
manifest = json.loads((OUTPUT / "fulltext" / "manifest.json").read_text())
print("Pinned model:", manifest["model"]["resolved_revision"])
print("Dataset:", manifest["dataset_revision_status"], manifest["corpus_sha256"])
print("Index:", manifest["index"])
print("Cache:", manifest["cache"])

In [ ]:
# Reuse embeddings and FAISS's persisted index in a fresh result directory.
cached_metrics = dense_experiment(
    corpus,
    queries,
    OUTPUT / "fulltext-cached",
    cache=CACHE,
    config=config,
    baseline=BASELINE,
)
assert metrics.equals(cached_metrics)
assert (OUTPUT / "fulltext" / "run.trec").read_bytes() == (
    OUTPUT / "fulltext-cached" / "run.trec"
).read_bytes()
cached_manifest = json.loads((OUTPUT / "fulltext-cached" / "manifest.json").read_text())
assert cached_manifest["cache"]["document_hit"] and cached_manifest["cache"]["query_hit"]
cached_manifest["cache"]